In [11]:
import os
import pandas as pd
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog, commonplayerinfo
from datetime import datetime
import time
from tqdm import tqdm

# Helper functions
def height_to_inches(height_str):
    try:
        feet, inches = map(int, height_str.split('-'))
        return feet * 12 + inches
    except:
        return None

def position_to_numeric(position):
    position_map = {'G': 0, 'F': 1, 'C': 2}
    if isinstance(position, str) and position:
        return position_map.get(position[0], -1)
    return -1

def infer_season(date):
    year = date.year
    return f"{year}-{year+1}" if date.month >= 9 else f"{year-1}-{year}"

# Load injury data once
injury_df = pd.read_csv("/Users/tymandachit/Documents/SW/MachineLearningProject/player_injures_2010-2025.csv")
injury_df['Date'] = pd.to_datetime(injury_df['Date'], errors='coerce').dt.normalize()
injury_df['Relinquished'] = injury_df['Relinquished'].astype(str).str.strip().str.lower()

# Cache player metadata
player_info_cache = {}

def get_injury_count(player_name, up_to_date):
    name_clean = player_name.strip().lower()
    return injury_df[
        (injury_df['Relinquished'] == name_clean) &
        (injury_df['Date'] <= up_to_date)
    ].shape[0]

def fetch_player_metadata(player_id):
    if player_id in player_info_cache:
        return player_info_cache[player_id]
    try:
        info_df = commonplayerinfo.CommonPlayerInfo(player_id=player_id).get_data_frames()[0]
        height_inches = height_to_inches(info_df.loc[0, 'HEIGHT'])
        weight = info_df.loc[0, 'WEIGHT']
        position_encoded = position_to_numeric(info_df.loc[0, 'POSITION'])
    except:
        height_inches, weight, position_encoded = None, None, -1
    player_info_cache[player_id] = (height_inches, weight, position_encoded)
    return height_inches, weight, position_encoded

def build_player_week_snapshots(player_name, season='2023-24'):
    player_info_list = players.find_players_by_full_name(player_name)
    if not player_info_list:
        print(f"No player found: {player_name}")
        return None
    player_id = player_info_list[0]['id']

    try:
        gamelog = playergamelog.PlayerGameLog(player_id=player_id, season=season, season_type_all_star='Regular Season')
        games_df = gamelog.get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching games for {player_name}: {e}")
        return None

    if games_df.empty:
        return None

    games_df['GAME_DATE'] = pd.to_datetime(games_df['GAME_DATE'], errors='coerce')
    games_df['season'] = games_df['GAME_DATE'].apply(infer_season)
    games_df = games_df.sort_values('GAME_DATE').reset_index(drop=True)

    snapshots = []
    weekly_cutoffs = pd.date_range(start=games_df['GAME_DATE'].min(), end=games_df['GAME_DATE'].max(), freq='W-MON')

    height_inches, weight, position_encoded = fetch_player_metadata(player_id)

    for cutoff in weekly_cutoffs:
        prior_games = games_df[games_df['GAME_DATE'] <= cutoff].copy()
        if prior_games.empty:
            continue

        prior_games['days_since_last_game'] = prior_games['GAME_DATE'].diff().dt.days
        back_to_backs = (prior_games['days_since_last_game'] == 1).sum()
        snapshot = {
            'player_name': player_name,
            'games_played': len(prior_games),
            'avg_minutes': prior_games['MIN'].mean(),
            'avg_points': prior_games['PTS'].mean(),
            'avg_assists': prior_games['AST'].mean(),
            'avg_rebounds': prior_games['REB'].mean(),
            'rolling5_minutes': prior_games.tail(5)['MIN'].mean(),
            'rolling5_points': prior_games.tail(5)['PTS'].mean(),
            'back_to_back_games': back_to_backs,
            'injury_count': get_injury_count(player_name, cutoff),
            'height_inches': height_inches,
            'weight': weight,
            'position_encoded': position_encoded,
            'season': season,
            'cutoff_date': cutoff
        }
        snapshots.append(snapshot)

    return pd.DataFrame(snapshots)

# Run script
if __name__ == "__main__":
    seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2010, 2025)]
    all_data = []

    for season in tqdm(seasons, desc="Processing Seasons"):
        df = build_player_week_snapshots("LeBron James", season)
        if df is not None:
            all_data.append(df)
        time.sleep(0.5)  # Respect API rate limit

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        os.makedirs("LivePlayerData", exist_ok=True)
        final_df.to_csv("LivePlayerData/lebron_james_all_seasons.csv", index=False)
        print("✅ Data saved.")


Processing Players:   0%|          | 10/5024 [00:38<5:32:01,  3.97s/it]